In [81]:
import pandas as pd
import numpy as np
import time
from openpyxl import load_workbook
from tqdm import tqdm

def carregar_bases():
    # Lista com caminhos dos arquivos de bases
    arquivos = [
        r"bases\GeoPlan.xlsx",
        r"bases\InfraGest.xlsx",
        r"bases\SisLic.xlsx",
        r"bases\t_conectados_anterior.xlsx",
        r"bases\t_conectados_atual.xlsx",
    ]
    
    # Lê as bases
    df_geoPlan, df_infraGest, df_sisLic, df_conectadosAnterior, df_conectadosAtual = [pd.read_excel(arq) for arq in arquivos]

    # Encontra o nome de cada aba no arquivo AuditReport 
    report_file = load_workbook(r"bases\AuditReport.xlsx")
    report_sheet1 = report_file.sheetnames[0]
    report_sheet2 = report_file.sheetnames[1]

    # Aba 1 do arquivo AuditReport
    df_report1 = pd.read_excel(r"bases\AuditReport.xlsx", sheet_name=report_sheet1)
    # Aba 2 do arquivo AuditReport
    df_report2 = pd.read_excel(r"bases\AuditReport.xlsx", sheet_name=report_sheet2)

    # Concatena as 2 abas do arquivo AuditReport
    df_report = pd.concat([df_report1, df_report2])

    # Faz merge do conectadosAtual com conectadosAntigo
    df_conectadosAtual = pd.DataFrame(df_conectadosAtual['ID_ESTACAO'])
    df_conectados = df_conectadosAtual.merge(df_conectadosAnterior, on='ID_ESTACAO', how='left')

    return df_geoPlan, df_infraGest, df_sisLic, df_conectados, df_report

df_geoPlan, df_infraGest, df_sisLic, df_conectados, df_report = carregar_bases()

In [ ]:
# juntar as colunas de todas as bases e transformar tudo em um. limpar sislic antes
def juntar_bases(df_geoPlan, df_infraGest, df_sisLic, df_conectados, df_report):
    # Filtrar colunas das bases
    df_geoPlan = df_geoPlan[['ID_ESTACAO', 'TIPO_ESTACAO_GEOPLAN']]
    df_infraGest = df_infraGest[['PONTO_ER', 'STATUS_INFRAGEST']]
    df_sisLic = df_sisLic[['CODIGO_ER', 'STATUS_SISLIC']]
    df_report = df_report[['CODIGO_ER 1', 'TIPO_DE_PONTO 62', 'LATITUDE 25', 'LONGITUDE 27']]

    # Prefixo AUX_
    df_geoPlan = df_geoPlan.rename(columns={'TIPO_ESTACAO_GEOPLAN': 'AUX_TIPO_ESTACAO_GEOPLAN', 'LATITUDE': 'AUX_LATITUDE', 'LONGITUDE': 'AUX_LONGITUDE'})
    df_infraGest = df_infraGest.rename(columns={'STATUS_INFRAGEST': 'AUX_STATUS_INFRAGEST'})
    df_report = df_report.rename(columns={'TIPO_DE_PONTO 62': 'AUX_TIPO_DE_PONTO', 'LATITUDE 25': 'AUX_LATITUDE', 'LONGITUDE 27': 'AUX_LONGITUDE'})

    # juntar bases conectados
    df_conectados = pd.merge(df_conectados, df_geoPlan, on= 'ID_ESTACAO', how= 'left')
    df_conectados = pd.merge(df_conectados, df_infraGest, left_on= 'ID_ESTACAO', right_on= 'PONTO_ER', how= 'left')
    df_conectados = pd.merge(df_conectados, df_report, left_on= 'ID_ESTACAO', right_on= 'CODIGO_ER 1', how= 'left')

    df_conectados = df_conectados.drop(columns=['PONTO_ER', 'CODIGO_ER 1'], errors='ignore')
    
    return df_conectados, df_sisLic

df_conectados, df_sisLic = juntar_bases(df_geoPlan, df_infraGest, df_sisLic, df_conectados, df_report)

,ID_ESTACAO,STATUS CONSOLIDADO,CHECKS,MES_REFERENCIA,STATUS E-MAIL,STATUS_SISLIC,TIPO_ESTACAO_GEOPLAN,SISTEMA_ER,TIPO_DE_PONTO 62,LATITUDE,...,GESTOR_RESPONSAVEL,CUSTO_MANUTENCAO_BRL,DATA_ULTIMA_VISTORIA,NIVEL_PRIORIDADE,OBSERVACOES_GERAIS,AUX_TIPO_ESTACAO_GEOPLAN,AUX_STATUS_INFRAGEST,AUX_TIPO_DE_PONTO,AUX_LATITUDE,AUX_LONGITUDE
0,EV-10000,SEM CONEXÃO DE REDE,OK,02/2026,ERRO,EM ANÁLISE,SHOPPING,PENDENTE ATIVAÇÃO,COBERTURA,-14.432900,...,Ana Costa,10532.68,2024-01-14,MÉDIA,Equipamento oxidado.,NaN,NaN,NaN,NaN,NaN
1,EV-10001,OPERACIONAL,OK,02/2026,ENVIADO,NÃO TEM LICENÇA,SHOPPING,INFRA NÃO CAPACITADA,POSTE,-20.491348,...,Ana Costa,1788.10,2025-04-23,MÉDIA,Equipamento oxidado.,NaN,NaN,POSTE,-20.991348,-47.176499
2,EV-10002,SEM CONEXÃO DE REDE,OK,02/2026,ENVIADO,APROVADO,INTERNO,CONECTADO,A DEFINIR,-28.479917,...,Mariana Souza,18710.99,2024-01-27,CRÍTICA,Revisar no próximo ciclo.,COBERTURA,INFRA NÃO CAPACITADA,TORRE,-28.479917,-52.302855
3,EV-10002,SEM CONEXÃO DE REDE,OK,02/2026,ENVIADO,APROVADO,INTERNO,CONECTADO,A DEFINIR,-28.479917,...,Mariana Souza,18710.99,2024-01-27,CRÍTICA,Revisar no próximo ciclo.,COBERTURA,CONECTADO,TORRE,-28.479917,-52.302855
4,EV-10003,OPERACIONAL,OK,02/2026,ERRO,PENDENTE INSTALAÇÃO,PONTO ECOLÓGICO,FORA DO AR,COBERTURA,-21.225005,...,Carlos Silva,5190.02,2024-02-03,CRÍTICA,Sem pendências.,SUPERMERCADO,NaN,NaN,NaN,NaN


In [87]:
def tratar_geoPlan():
    pass

In [88]:
def tratar_sisLic():
        # tratar duplicidade

        # Caso encontre alguma chave com STATUS_SISLIC = APROVADO, exclui as outras duplicadas.
        # Se não tiver APROVADO, busca EM ANÁLISE e exclui as duplicadas restantes.

        # duplicated_keys = df_sisLic['CODIGO_ER'].duplicated()
        # unique_keys = ~df_sisLic['CODIGO_ER'].duplicated()

        # print(duplicated_keys)
        
        pass

In [89]:
def tratar_infraGest():
        pass

In [90]:
def classificar_ERs():
        pass

In [91]:
def definir_faturamento():
        pass

In [92]:
def main():
    print("Carregando bases...")
    carregar_bases()

    # juntar as colunas de todas as bases e transformar tudo em um. limpar sislic antes
    juntar_bases()

    # 2 - Regras da base GeoPlan:
    #     2.1 - Todos os valores nulos em geoplan['TIPO_ESTACAO_GEOPLAN'] recebem o valor A DEFINIR.
    #     2.2 - Se geoplan['TIPO_ESTACAO_GEOPLAN'] e audit_report['AUX_TIPO_PONTO_AUDIT'] tiverem o mesmo valor, nada é feito.
    #     2.3 - Se o valor na coluna geoplan['TIPO_ESTACAO_GEOPLAN'] for igual a A DEFINIR, mas não em audit_report['AUX_TIPO_PONTO_AUDIT'], então geoplan['TIPO_ESTACAO_GEOPLAN'] assume o valor de audit_report['AUX_TIPO_PONTO_AUDIT'].
    #     2.4 - Mantém os valores de audit_report['AUX_LATITUDE_AUDIT'] e audit_report['AUX_LONGITUDE_AUDIT'] e o que estiver vazio recebe as coordenadas de geoplan['LATITUDE'] e geoplan['LONGITUDE'] genéricas.
    tratar_geoPlan()

    # 3 - Regras da base SisLic:
    #     ** sislic tem chaves repetidas em sislic['CODIGO_ER'] com valores de licença diferentes que não devem ser excluídas de imediato.
    #     Antes da comparação:
    #         Caso encontre alguma chave com sislic['STATUS_SISLIC'] = APROVADO, exclui as outras chaves duplicadas.
    #         Se não tiver APROVADO, busca EM ANÁLISE e exclui as chaves duplicadas restantes.

    #     3.1 - Se o valor na coluna base_consolidada['STATUS_SISLIC'] e sislic['AUX_STATUS_SISLIC'] forem iguais, nada é feito.
    #     3.2 - Se o valor na coluna base_consolidada['STATUS_SISLIC'] for igual a APROVADO e sislic['AUX_STATUS_SISLIC'] não for APROVADO, envia para a tabela de validação.
    #     3.3 - Se o valor na coluna base_consolidada['STATUS_SISLIC'] for diferente de APROVADO, substitui o valor de base_consolidada['STATUS_SISLIC'] pelo valor de sislic['AUX_STATUS_SISLIC'].
    tratar_sisLic()

    # 4 - Regras da base InfraGest:
    #     4.1 - Se o valor em base_consolidada['SISTEMA_ER'] for igual a CONECTADO e infragest['AUX_STATUS_INFRAGEST'] for igual a CONECTADO, nada é feito.
    #     4.2 - Se o valor em base_consolidada['SISTEMA_ER'] estiver vazio, recebe o valor de infragest['AUX_STATUS_INFRAGEST'].
    #     4.3 - Se o valor em base_consolidada['SISTEMA_ER'] for igual a CONECTADO e infragest['AUX_STATUS_INFRAGEST'] for diferente de CONECTADO, envia para validação.
    #     4.4 - Se o valor em base_consolidada['SISTEMA_ER'] estiver vazio e o valor de infragest['AUX_STATUS_INFRAGEST'] também, então base_consolidada['SISTEMA_ER'] é igual a INFRA NÃO CAPACITADA.
    tratar_infraGest()

    # 5 - Regras para classificação de ERs:
    #     5.1 - Se o valor na coluna base_consolidada['STATUS E-MAIL'] estiver preenchido, mantém a informação de base_consolidada['STATUS CONSOLIDADO'].
    #     5.2 - Se geoplan['TIPO_ESTACAO_GEOPLAN'] for igual a SHOPPING ou SUPERMERCADO ou PONTO ECOLÓGICO, então base_consolidada['STATUS CONSOLIDADO'] é igual a ESTAÇÕES SUSTENTÁVEIS (ENERGIA SOLAR).
    #     5.3 - Se geoplan['TIPO_ESTACAO_GEOPLAN'] for igual a INTERNO, então base_consolidada['STATUS CONSOLIDADO'] é igual a CARREGADOR LENTO - INTERNO.
    #     5.4 - Se geoplan['TIPO_ESTACAO_GEOPLAN'] for igual a COBERTURA, então base_consolidada['STATUS CONSOLIDADO'] é igual a ESTAÇÃO EM COBERTURA.
    #     5.5 - Se geoplan['TIPO_ESTACAO_GEOPLAN'] for igual a A DEFINIR ou NOVO_TERRENO ou COMERCIAL e sislic['STATUS_SISLIC'] for igual a APROVADO, então base_consolidada['STATUS CONSOLIDADO'] é igual a OPERACIONAL.
    #     5.6 - Se geoplan['TIPO_ESTACAO_GEOPLAN'] for igual a A DEFINIR ou NOVO_TERRENO ou COMERCIAL e sislic['STATUS_SISLIC'] for igual a NÃO TEM LICENÇA ou PENDENTE INSTALAÇÃO, então base_consolidada['STATUS CONSOLIDADO'] mantém os status atuais e nos campos vazios muda para "SEM CONEXÃO DE REDE".
    classificar_ERs()

    # 6 - Regras da coluna STATUS_FATURAMENTO:
    #     6.1 - Se base_consolidada['STATUS CONSOLIDADO'] for igual a OPERACIONAL e infragest['SISTEMA_ER'] for igual a CONECTADO ou PENDENTE ATIVAÇÃO, então base_consolidada['STATUS_FATURAMENTO'] é igual a 1.
    #     6.2 - Se base_consolidada['STATUS CONSOLIDADO'] for igual a OPERACIONAL e infragest['SISTEMA_ER'] for igual a INFRA NÃO CAPACITADA, então base_consolidada['STATUS_FATURAMENTO'] é igual a 0.
    #     6.3 - Se qualquer valor não bater com uma das duas condições anteriores, então base_consolidada['STATUS_FATURAMENTO'] é igual a 2.
    definir_faturamento()

    pass